In [1]:
import os
from pathlib import Path

import librosa
import soundfile as sf
import numpy as np
import pandas as pd

from tqdm import tqdm
from audiomentations import Compose
from audiomentations import AddGaussianNoise
from audiomentations import PitchShift
from audiomentations import TimeStretch
from audiomentations import Shift

In [2]:
train = pd.read_csv("C:/UserData/Kaustav/python projects/GitaVani/dataset/processed/train.csv")
val = pd.read_csv("C:/UserData/Kaustav/python projects/GitaVani/dataset/processed/val.csv")
test = pd.read_csv("C:/UserData/Kaustav/python projects/GitaVani/dataset/processed/test.csv")
print(len(train))
print(len(val))
print(len(test))

7680
1624
1700


In [3]:
OUTPUT = Path("C:/UserData/Kaustav/python projects/GitaVani/dataset/processed/audio")

train_dir = OUTPUT / "train"
val_dir = OUTPUT / "val"
test_dir = OUTPUT / "test"
train_dir.mkdir(parents=True, exist_ok=True)
val_dir.mkdir(parents=True, exist_ok=True)
test_dir.mkdir(parents=True, exist_ok=True)

In [4]:
TARGET_SR = 16000
DURATION = 3
MAX_LENGTH = TARGET_SR * DURATION

In [5]:
sample = train.iloc[0]["filepath"]
print(sample)

C:\UserData\Kaustav\python projects\GitaVani\dataset\raw\ravdess\archive\Actor_01\03-01-02-01-01-01-01.wav


In [6]:
audio, sr = librosa.load(
    sample,
    sr=TARGET_SR,
    mono=True
)
print(sr)
print(audio.shape)
print(len(audio) / sr)

16000
(56590,)
3.536875


In [7]:
def fix_length(audio):
    if len(audio) > MAX_LENGTH:
        audio = audio[:MAX_LENGTH]
    else:
        padding = MAX_LENGTH - len(audio)
        audio = np.pad(audio, (0, padding))
    return audio

In [8]:
audio = fix_length(audio)
print(len(audio))

48000


In [9]:
def normalize(audio):
    max_val = np.max(np.abs(audio))
    if max_val > 0:
        audio = audio / max_val
    return audio

In [10]:
audio = normalize(audio)
print(audio.min())
print(audio.max())

-0.98127615
1.0


In [11]:
save_path = train_dir / "sample.wav"
sf.write(
    save_path,
    audio,
    TARGET_SR
)

In [12]:
def preprocess_audio(filepath):
    audio, sr = librosa.load(
        filepath,
        sr=TARGET_SR,
        mono=True
    )
    audio = fix_length(audio)
    audio = normalize(audio)
    return audio

In [13]:
audio = preprocess_audio(sample)
print(audio.shape)

(48000,)


In [14]:
for _, row in tqdm(val.iterrows(), total=len(val)):
    audio = preprocess_audio(row["filepath"])
    filename = Path(row["filepath"]).name
    sf.write(
        val_dir / filename,
        audio,
        TARGET_SR
    )

100%|██████████| 1624/1624 [00:07<00:00, 207.53it/s]


In [15]:
for _, row in tqdm(test.iterrows(), total=len(test)):
    audio = preprocess_audio(row["filepath"])
    filename = Path(row["filepath"]).name
    sf.write(
        test_dir / filename,
        audio,
        TARGET_SR
    )

100%|██████████| 1700/1700 [00:09<00:00, 171.67it/s]


In [ ]:
print(len(list(val_dir.glob("*.wav"))))
print(len(list(test_dir.glob("*.wav"))))

1462
1500


In [17]:
augment = Compose([
    AddGaussianNoise(
        min_amplitude=0.001,
        max_amplitude=0.01,
        p=0.5
    ),
    PitchShift(
        min_semitones=-2,
        max_semitones=2,
        p=0.5
    ),
    TimeStretch(
        min_rate=0.9,
        max_rate=1.1,
        p=0.5
    ),
    Shift(
        min_shift=-0.1,
        max_shift=0.1,
        p=0.5
    )
])

In [18]:
calm = train[
    train["emotion"] == "calm"
]

In [19]:
print(len(calm))

272


In [20]:
sample = calm.iloc[0]["filepath"]
audio = preprocess_audio(sample)
aug_audio = augment(
    samples=audio,
    sample_rate=TARGET_SR
)

In [21]:
from IPython.display import Audio
print("Original")
display(Audio(audio, rate=TARGET_SR))
print("Augmented")
display(Audio(aug_audio, rate=TARGET_SR))

Original


Augmented


In [23]:
sf.write(
    train_dir / "aug_test.wav",
    aug_audio,
    TARGET_SR
)
print((train_dir / "aug_test.wav").exists())

True


In [ ]:
for _, row in tqdm(calm.iterrows(), total=len(calm)):
    audio = preprocess_audio(row["filepath"])
    filename = Path(row["filepath"]).stem
    # Save original
    sf.write(
        train_dir / f"{filename}.wav",
        audio,
        TARGET_SR
    )
    # Generate four augmented versions
    for i in range(4):

        aug_audio = augment(
            samples=audio,
            sample_rate=TARGET_SR
        )
        sf.write(
            train_dir / f"{filename}_aug{i}.wav",
            aug_audio,
            TARGET_SR
        )

100%|██████████| 272/272 [00:17<00:00, 15.98it/s]


In [25]:
others = train[
    train["emotion"] != "calm"
]

for _, row in tqdm(others.iterrows(), total=len(others)):

    audio = preprocess_audio(row["filepath"])

    filename = Path(row["filepath"]).name

    sf.write(
        train_dir / filename,
        audio,
        TARGET_SR
    )

100%|██████████| 7408/7408 [00:39<00:00, 189.43it/s]


In [26]:
print("Train audio:", len(list(train_dir.glob("*.wav"))))
print("Validation audio:", len(list(val_dir.glob("*.wav"))))
print("Test audio:", len(list(test_dir.glob("*.wav"))))

Train audio: 6558
Validation audio: 1462
Test audio: 1500
